In [ ]:
%pip install openai beautifulsoup4

# Workload Profiler — Pay-Per-Token

Profiles **per-request** token and latency characteristics against a PPT endpoint. Use the output as inputs to the ITPM/OTPM calculation and the Databricks GenAI Pricing Calculator.

| Metric | Description | Used for |
|---|---|---|
| **Input tokens** | Prompt tokens per request | `ITPM = input_tokens × QPM` |
| **Output tokens** | Completion tokens per request | `OTPM = output_tokens × QPM` |
| **TTFT** | Time to first token (ms) | Latency baseline |
| **TPOT** | Time per output token (ms) | Latency baseline |

**Note:** This notebook cannot calculate ITPM or OTPM directly — those require QPM, which comes from your production traffic or estimation. See the README for how to derive QPM for your workload type.

In [ ]:
import os

os.environ["DATABRICKS_HOST"]  = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
os.environ["DATABRICKS_TOKEN"] = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

# Change to the pay-per-token model you want to profile
PPT_MODEL = "databricks-gpt-oss-20b"

# PPT rate limits sourced from:
# https://docs.databricks.com/aws/en/machine-learning/foundation-model-apis/limits
# QPH = queries per hour (None = not published). Update as limits change.
PPT_LIMITS = {
    # GPT OSS
    "databricks-gpt-oss-120b":              {"itpm": 200_000, "otpm": 10_000, "qph": None},
    "databricks-gpt-oss-20b":               {"itpm": 200_000, "otpm": 10_000, "qph": None},
    # Meta Llama
    "databricks-meta-llama-4-maverick":     {"itpm": 200_000, "otpm": 10_000, "qph": 2_400},
    "databricks-meta-llama-3-3-70b-instruct": {"itpm": 200_000, "otpm": 10_000, "qph": 2_400},
    "databricks-meta-llama-3-1-8b-instruct":  {"itpm": 200_000, "otpm": 10_000, "qph": 7_200},
    "databricks-meta-llama-3-1-405b-instruct": {"itpm": 5_000,  "otpm": 500,    "qph": 1_200},
    # Google
    "databricks-gemma-3-12b":              {"itpm": 200_000, "otpm": 10_000, "qph": 7_200},
    "databricks-gemini-2-5-pro":           {"itpm": 200_000, "otpm": 20_000, "qph": 360_000},
    "databricks-gemini-2-5-flash":         {"itpm": 200_000, "otpm": 20_000, "qph": 360_000},
    # Anthropic Claude
    "databricks-claude-sonnet-4":          {"itpm": 200_000, "otpm": 20_000, "qph": 360_000},
    "databricks-claude-opus-4-7":          {"itpm": 200_000, "otpm": 20_000, "qph": 360_000},
    "databricks-claude-haiku-4-5":         {"itpm": 200_000, "otpm": 20_000, "qph": 360_000},
}

In [ ]:
import requests
from bs4 import BeautifulSoup

# Maps docs display names → API model names.
# Only the limits numbers need dynamic fetching; this mapping changes rarely.
_DISPLAY_TO_API = {
    "GPT OSS 120B":             "databricks-gpt-oss-120b",
    "GPT OSS 20B":              "databricks-gpt-oss-20b",
    "Llama 4 Maverick":         "databricks-meta-llama-4-maverick",
    "Llama 3.3 70B Instruct":   "databricks-meta-llama-3-3-70b-instruct",
    "Llama 3.1 8B Instruct":    "databricks-meta-llama-3-1-8b-instruct",
    "Llama 3.1 405B Instruct":  "databricks-meta-llama-3-1-405b-instruct",
    "Gemma 3 12B":              "databricks-gemma-3-12b",
    "Gemini 2.5 Pro":           "databricks-gemini-2-5-pro",
    "Gemini 2.5 Flash":         "databricks-gemini-2-5-flash",
    "Claude Sonnet 4":          "databricks-claude-sonnet-4",
    "Claude Opus 4.7":          "databricks-claude-opus-4-7",
    "Claude Haiku 4.5":         "databricks-claude-haiku-4-5",
}

def _parse_int(s: str) -> int | None:
    s = s.replace(",", "").strip()
    return int(s) if s.isdigit() else None

def _fetch_limits() -> dict | None:
    try:
        resp = requests.get(
            "https://docs.databricks.com/aws/en/machine-learning/foundation-model-apis/limits",
            timeout=10,
        )
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        limits = {}
        for table in soup.find_all("table"):
            headers = [th.get_text(strip=True).upper() for th in table.find_all("th")]
            if not {"ITPM", "OTPM"}.issubset(headers):
                continue
            for row in table.find_all("tr")[1:]:
                cells = [td.get_text(strip=True) for td in row.find_all("td")]
                if not cells:
                    continue
                row_data = dict(zip(headers, cells))
                api_name = _DISPLAY_TO_API.get(row_data.get("MODEL", ""))
                if api_name:
                    limits[api_name] = {
                        "itpm": _parse_int(row_data.get("ITPM", "")),
                        "otpm": _parse_int(row_data.get("OTPM", "")),
                        "qph":  _parse_int(row_data.get("QPH", "")),
                    }

        if limits:
            print(f"Fetched limits for {len(limits)} models from docs.")
            return limits
        print("Warning: parsed 0 models from docs page — using static fallback.")
        return None

    except Exception as e:
        print(f"Warning: could not fetch limits ({e}) — using static fallback.")
        return None

ppt_limits = _fetch_limits() or PPT_LIMITS
print(f"Active limits source: {'docs' if ppt_limits is not PPT_LIMITS else 'static fallback'}")

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url=f"{os.environ['DATABRICKS_HOST']}/serving-endpoints",
    api_key=os.environ["DATABRICKS_TOKEN"],
)

In [ ]:
import time


def profile_request(messages: list[dict]) -> dict:
    """
    Two calls per query:
    1. Non-streaming  → input/output token counts (usage is in the response object)
    2. Streaming      → TTFT and TPOT (Databricks API does not support stream_options)

    TTFT = time from request start → first token received
    TPOT = (total_time - TTFT) / (output_tokens - 1)
    """
    # --- token counts ---
    response = client.chat.completions.create(
        model=PPT_MODEL,
        messages=messages,
    )
    input_tokens  = response.usage.prompt_tokens
    output_tokens = response.usage.completion_tokens

    # --- latency metrics ---
    ttft_ms = None
    t_start = time.perf_counter()

    stream = client.chat.completions.create(
        model=PPT_MODEL,
        messages=messages,
        stream=True,
    )

    for chunk in stream:
        if ttft_ms is None and chunk.choices and chunk.choices[0].delta.content:
            ttft_ms = (time.perf_counter() - t_start) * 1000

    total_ms = (time.perf_counter() - t_start) * 1000
    tpot_ms  = (total_ms - ttft_ms) / (output_tokens - 1) if output_tokens > 1 else 0.0

    return {
        "input_tokens":  input_tokens,
        "output_tokens": output_tokens,
        "ttft_ms":       round(ttft_ms, 2),
        "tpot_ms":       round(tpot_ms, 2),
        "total_ms":      round(total_ms, 2),
    }

## Sample queries

Replace or extend `SAMPLE_QUERIES` with prompts representative of your actual workload.
The goal is to capture a realistic distribution of input and output lengths.

In [ ]:
SAMPLE_QUERIES = [
    "What is machine learning? Explain in 3 paragraphs.",
    "Summarize the key differences between supervised and unsupervised learning.",
    "Write a Python function that reads a CSV file and returns the top 5 rows.",
    "What is the capital of France?",
    "Explain transformer architecture in detail, covering attention mechanisms, positional encoding, and the encoder-decoder structure.",
]

results = []

for i, query in enumerate(SAMPLE_QUERIES):
    print(f"[{i+1}/{len(SAMPLE_QUERIES)}] {query[:60]}...")
    result = profile_request([{"role": "user", "content": query}])
    results.append({"query": query, **result})
    print(f"  input={result['input_tokens']} output={result['output_tokens']} "
          f"ttft={result['ttft_ms']}ms tpot={result['tpot_ms']}ms\n")

In [ ]:
import statistics

def percentile(data: list[float], p: int) -> float:
    sorted_data = sorted(data)
    idx = int(len(sorted_data) * p / 100)
    return round(sorted_data[min(idx, len(sorted_data) - 1)], 2)

def summarize(key: str) -> dict:
    vals = [r[key] for r in results]
    return {
        "avg":  round(statistics.mean(vals), 2),
        "p50":  percentile(vals, 50),
        "p75":  percentile(vals, 75),
        "p95":  percentile(vals, 95),
        "max":  round(max(vals), 2),
    }

metrics = ["input_tokens", "output_tokens", "ttft_ms", "tpot_ms"]
labels  = ["Input tokens", "Output tokens", "TTFT (ms)", "TPOT (ms)"]

print(f"{'Metric':<18} {'avg':>8} {'p50':>8} {'p75':>8} {'p95':>8} {'max':>8}")
print("-" * 58)
for metric, label in zip(metrics, labels):
    s = summarize(metric)
    print(f"{label:<18} {s['avg']:>8} {s['p50']:>8} {s['p75']:>8} {s['p95']:>8} {s['max']:>8}")

print(f"\nModel: {PPT_MODEL}  |  n={len(results)} requests")
print("\nUse avg input/output tokens and QPM from your production traffic")
print("in the Databricks GenAI Pricing Calculator to size your PT endpoint.")

## PPT Capacity Ceiling for This Workload

If you don't have historical QPM data, you can reverse-calculate the maximum QPM PPT can serve
given your measured request shape. This tells you whether PPT is sufficient before you ever see
production traffic.

```
max_qpm_from_itpm = PPT_ITPM_CEILING / avg_input_tokens
max_qpm_from_otpm = PPT_OTPM_CEILING / avg_output_tokens
ppt_max_qpm       = min(max_qpm_from_itpm, max_qpm_from_otpm)   ← binding constraint
```

If your expected QPM will exceed `ppt_max_qpm`, PT is required.

In [ ]:
import statistics

avg_input  = statistics.mean(r["input_tokens"]  for r in results)
avg_output = statistics.mean(r["output_tokens"] for r in results)

limits = ppt_limits.get(PPT_MODEL)
if limits is None:
    print(f"Warning: {PPT_MODEL} not found in limits — add it to _DISPLAY_TO_API or PPT_LIMITS.")
else:
    itpm = limits["itpm"]
    otpm = limits["otpm"]
    qph  = limits["qph"]

    constraints = {
        "ITPM": itpm / avg_input,
        "OTPM": otpm / avg_output,
    }
    if qph:
        constraints["QPH"] = qph / 60

    binding     = min(constraints, key=constraints.get)
    ppt_max_qpm = constraints[binding]

    print(f"Avg input tokens/request  : {avg_input:.1f}")
    print(f"Avg output tokens/request : {avg_output:.1f}")
    print()
    print(f"  ITPM ceiling ({itpm:>7,}) → {constraints['ITPM']:.1f} QPM")
    print(f"  OTPM ceiling ({otpm:>7,}) → {constraints['OTPM']:.1f} QPM")
    if qph:
        print(f"  QPH  ceiling ({qph:>7,}) → {constraints['QPH']:.1f} QPM  ({qph:,} QPH)")
    print()
    print(f"  Binding constraint : {binding}")
    print(f"  PPT max QPM        : {ppt_max_qpm:.1f} QPM  ({ppt_max_qpm / 60:.2f} QPS)")
    print()
    print("→ If your expected QPM exceeds this, PPT cannot serve your load. Move to PT.")